### Исследование использование ранга факторных матриц в качестве тензорного ранга в разложении Такера

In [ ]:
import torch
import numpy as np
import pandas as pd
import tensorly as tl
from tensorly.tenalg import multi_mode_dot
from tensorly.decomposition import parafac, randomised_parafac
tl.set_backend('pytorch')
import matplotlib.pyplot as plt
import random

In [2]:
def generate_low_tucker_rank_tensor(
    shape: tuple[int, ...], 
    rank: tuple[int, ...], 
    device=None
) -> tuple[torch.Tensor, torch.Tensor, list[torch.Tensor]]:
    
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. Генерируем случайные ортонормированные фактор-матрицы
    # Столбцы каждой матрицы U_i будут ортонормированным базисом
    factor_matrices = []
    for s, r in zip(shape, rank):
        # Создаем случайную матрицу и получаем ее ортонормированный базис через QR-разложение
        # Это гарантирует, что U.T @ U будет единичной матрицей
        q, _ = torch.linalg.qr(torch.randn(s, r, device=device))
        factor_matrices.append(q)
        
    # 2. Генерируем случайное ядро (core tensor)
    core_tensor = torch.randn(*rank, device=device)
    
    # 3. Собираем полный тензор через мульти-тензорное произведение
    low_rank_tensor = multi_mode_dot(core_tensor, factor_matrices)
    
    return low_rank_tensor, core_tensor, factor_matrices


#### Поиск ранга

### Тензоры размеров n^d, где n - степени двойки

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# device = 'cpu'
epsilon = 1e-7  # Уровень шума 
n_samples = 1000  # Сэмплы для randomised_parafac

num_repeats_one_tensor = 3  # Повторов для одного и того же тензора (усреднение шума измерений)
num_tensors = 5            # Количество разных тензоров одного размера (усреднение по данным)
max_log = 9                # Максимальная степень двойки
dims = [3, 4]              # Размерности (порядок тензора)

results_summary = [] # Для усредненных данных
results_raw = []     # Для сырых данных всех запусков

print(f"Start computation on: {device}")
print(f"Testing dimensions: {dims}")
print(f"Sizes 2^1 to 2^{max_log}")

for d in dims:
    # Проход по размерам тензора: 2, 4, 8, ..., 2^max_log
    sizes = [2**i for i in range(1, max_log + 1)]
    
    for n in sizes:
        print(f"--> Processing Order-{d} tensor of size {n}x...x{n}")
        
        # Накопители для усреднения по категории (размер n, порядок d)
        agg_time_cp = []
        agg_time_svd = []
        agg_rank_err_cp = []
        agg_rank_err_svd = []
        
        for t_idx in range(num_tensors):
            tensor_shape = tuple([n for _ in range(d)])
            tucker_rank_true = tuple([random.randint(1, n) for _ in range(d)])
            
            try:
                B_cpu, _, _ = generate_low_tucker_rank_tensor(tensor_shape, tucker_rank_true)
                # noise = torch.randn_like(B_cpu) * epsilon * tl.norm(B_cpu)
                # B_cpu = B_cpu + noise
                
                B = tl.tensor(B_cpu, device=device)
        
                cp_rank = max(tensor_shape)
                
            except Exception as e:
                print(f"Error generating {tensor_shape}: {e}")
                continue

            # Вектор истинного ранга для расчета ошибки
            vec_true = torch.tensor(tucker_rank_true, dtype=torch.float32, device='cpu')

            # Повторные запуски алгоритмов на одном тензоре
            for r_idx in range(num_repeats_one_tensor):
                run_stat = {
                    "Order": d,
                    "Size": n,
                    "Tensor_ID": t_idx,
                    "True_Rank": str(tucker_rank_true)
                }
                
                # CP
                start_event = torch.cuda.Event(enable_timing=True)
                end_event = torch.cuda.Event(enable_timing=True)
                
                time_cp_total = np.nan
                rank_err_cp = np.nan
                calc_rank_cp = None
                
                try:
                    start_event.record()
                    
                    # Разложение
                    factors_obj = randomised_parafac(
                        B, 
                        rank=max(tensor_shape), 
                        n_samples=n_samples, 
                        tol=1e-3, 
                        verbose=0
                    )
                    
                    if hasattr(factors_obj, 'factors'):
                        factors = factors_obj.factors
                    else:
                        # (weights, factors)
                        factors = factors_obj[1]

                    current_calc_rank = []
                    for factor_matrix in factors:
                        # factor_matrix shape: (Size, CP_Rank)
                        # Ранг матрицы <= min(Size, CP_Rank)
                        r_approx = torch.linalg.matrix_rank(factor_matrix).item()
                        current_calc_rank.append(r_approx)
                    
                    calc_rank_cp = tuple(current_calc_rank)
                    
                    end_event.record()
                    torch.cuda.synchronize()
                    time_cp_total = start_event.elapsed_time(end_event) / 1000.0
                    
                    # Ошибка ранга CP
                    vec_calc_cp = torch.tensor(calc_rank_cp, dtype=torch.float32, device='cpu')
                    rank_err_cp = (torch.norm(vec_true - vec_calc_cp) / torch.norm(vec_true)).item()
                    
                except RuntimeError as e:
                    if "out of memory" in str(e):
                        torch.cuda.empty_cache()
                    else:
                        print(f"CP Error: {e}")

                # SVD
                start_event_svd = torch.cuda.Event(enable_timing=True)
                end_event_svd = torch.cuda.Event(enable_timing=True)
                
                time_svd_total = np.nan
                rank_err_svd = np.nan
                calc_rank_svd = None

                try:
                    start_event_svd.record()
                    
                    current_svd_rank = []
                    for mode in range(d):
                        unfolded = tl.unfold(B, mode)
                        r_mode = torch.linalg.matrix_rank(unfolded).item()
                        current_svd_rank.append(r_mode)
                    
                    calc_rank_svd = tuple(current_svd_rank)
                    
                    end_event_svd.record()
                    torch.cuda.synchronize()
                    time_svd_total = start_event_svd.elapsed_time(end_event_svd) / 1000.0
                    
                    # Ошибка ранга SVD
                    vec_calc_svd = torch.tensor(calc_rank_svd, dtype=torch.float32, device='cpu')
                    rank_err_svd = (torch.norm(vec_true - vec_calc_svd) / torch.norm(vec_true)).item()
                    
                except RuntimeError as e:
                    if "out of memory" in str(e):
                        torch.cuda.empty_cache()
                    else: 
                        print(f"SVD Error: {e}")

                # Запись сырых данных
                run_stat.update({
                    "CP_Time": time_cp_total,
                    "CP_Rank_Est": str(calc_rank_cp),
                    "CP_Rank_Err": rank_err_cp,
                    "SVD_Time": time_svd_total,
                    "SVD_Rank_Est": str(calc_rank_svd),
                    "SVD_Rank_Err": rank_err_svd
                })
                results_raw.append(run_stat)
                
                # Добавляем в списки для усреднения (игнорируя NaN)
                if not np.isnan(time_cp_total): agg_time_cp.append(time_cp_total)
                if not np.isnan(rank_err_cp): agg_rank_err_cp.append(rank_err_cp)
                
                if not np.isnan(time_svd_total): agg_time_svd.append(time_svd_total)
                if not np.isnan(rank_err_svd): agg_rank_err_svd.append(rank_err_svd)

        # Агрегация по размеру N (после прохода всех тензоров и повторов)
        summary_stat = {
            "Dimension": d,
            "Size": n,
            "Avg_Time_CP": np.mean(agg_time_cp) if agg_time_cp else np.nan,
            "Std_Time_CP": np.std(agg_time_cp) if agg_time_cp else np.nan,
            "Avg_Rank_Err_CP": np.mean(agg_rank_err_cp) if agg_rank_err_cp else np.nan,
            
            "Avg_Time_SVD": np.mean(agg_time_svd) if agg_time_svd else np.nan,
            "Std_Time_SVD": np.std(agg_time_svd) if agg_time_svd else np.nan,
            "Avg_Rank_Err_SVD": np.mean(agg_rank_err_svd) if agg_rank_err_svd else np.nan,
            
            "Successful_Runs_CP": len(agg_time_cp),
            "Successful_Runs_SVD": len(agg_time_svd)
        }
        results_summary.append(summary_stat)
        
        print(f"Done {n}^({d}): "
              f"CP Time={summary_stat['Avg_Time_CP']:.4f}s, "
              f"SVD Time={summary_stat['Avg_Time_SVD']:.4f}s")
        
        # Очистка памяти перед следующим увеличением размера
        torch.cuda.empty_cache()

# Сохранение результатов
df_raw = pd.DataFrame(results_raw)
df_summary = pd.DataFrame(results_summary)

pd.options.display.float_format = '{:,.4f}'.format

df_summary.to_csv('nxd_summary.csv', sep=';', index=False)
df_raw.to_csv('nxd_all.csv', sep=';', index=False)

print("\n--- Summary Results ---")
print(df_summary[['Dimension', 'Size', 'Avg_Time_CP', 'Avg_Time_SVD', 'Avg_Rank_Err_CP', 'Avg_Rank_Err_SVD']].to_string())
print("\nFiles saved: 'tensor_rank_summary.csv', 'tensor_rank_raw_runs.csv'")


In [ ]:
import seaborn as sns

# Настройка стиля для академичности
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
plt.rcParams['font.family'] = 'sans-serif'

def plot_benchmark_results(file_path):
    """
    Загружает сводный CSV и строит графики сравнения времени CP и SVD.
    """
    # 1. Загрузка данных
    try:
        df = pd.read_csv(file_path, sep=';')
    except FileNotFoundError:
        print(f"Файл {file_path} не найден. Проверьте путь.")
        return

    # Проверка наличия необходимых колонок
    required_cols = ['Dimension', 'Size', 'Avg_Time_CP', 'Avg_Time_SVD']
    if not all(col in df.columns for col in required_cols):
        print(f"В файле отсутствуют необходимые колонки. Ожидаются: {required_cols}")
        print(f"Найдены: {df.columns.tolist()}")
        return

    unique_orders = sorted(df['Dimension'].unique())
    num_plots = len(unique_orders)

    # 2. Создание полотна графиков
    fig, axes = plt.subplots(1, num_plots, figsize=(6 * num_plots, 5), squeeze=False)
    axes = axes.flatten() # Чтобы удобно итерироваться, даже если график один

    for ax, order in zip(axes, unique_orders):
        print(f"--- Обработка Order d={order} ---")
        
        # Фильтрация данных для текущей размерности
        mask = (df['Dimension'] == order)
        data = df[mask].sort_values(by='Size')
        
        n_vals = data['Size'].to_numpy()
        
        # Данные CP
        time_cp = data['Avg_Time_CP'].to_numpy()
        std_cp = data['Std_Time_CP'].to_numpy() if 'Std_Time_CP' in data.columns else np.zeros_like(time_cp)
        
        # Данные SVD
        time_svd = data['Avg_Time_SVD'].to_numpy()
        std_svd = data['Std_Time_SVD'].to_numpy() if 'Std_Time_SVD' in data.columns else np.zeros_like(time_svd)
        
        # Очистка от NaN (если какой-то метод упал по OOM)
        # Строим только там, где есть валидные значения
        valid_cp = ~np.isnan(time_cp)
        valid_svd = ~np.isnan(time_svd)
        
        # --- Отрисовка CP ---
        if np.any(valid_cp):
            ax.plot(n_vals[valid_cp], time_cp[valid_cp], 'o-', 
                    color='#1f77b4', label='CP Decomposition', linewidth=2, markersize=6)
            ax.fill_between(n_vals[valid_cp], 
                            time_cp[valid_cp] - std_cp[valid_cp], 
                            time_cp[valid_cp] + std_cp[valid_cp], 
                            color='#1f77b4', alpha=0.2)
        
        # --- Отрисовка SVD ---
        if np.any(valid_svd):
            ax.plot(n_vals[valid_svd], time_svd[valid_svd], 's--', 
                    color='#ff7f0e', label='HOSVD (SVD-based)', linewidth=2, markersize=6)
            ax.fill_between(n_vals[valid_svd], 
                            time_svd[valid_svd] - std_svd[valid_svd], 
                            time_svd[valid_svd] + std_svd[valid_svd], 
                            color='#ff7f0e', alpha=0.2)

        # Логарифмическая шкала часто полезна для таких сравнений, 
        # но если хотите линейную - закомментируйте следующие строки
        ax.set_yscale('log')
        ax.set_xscale('log', base=2)
        
        # Оформление
        ax.set_title(f'Тензоры порядка $d={order}$', fontsize=14, fontweight='bold')
        ax.set_xlabel('Размер тензора $n$ (log scale)', fontsize=12)
        ax.set_ylabel('Время выполнения, сек (log scale)', fontsize=12)
        
        # Настройка тиков оси X (степени двойки)
        ax.set_xticks(n_vals)
        ax.set_xticklabels(n_vals)
        
        ax.grid(True, which="both", ls="-", alpha=0.2)
        ax.grid(True, which="major", ls="--", alpha=0.6)
        ax.legend(fontsize=10)

    plt.tight_layout()
    # plt.savefig('tensor_rank_benchmark.png', dpi=300) # Сохранить в файл
    plt.show()


In [ ]:
plot_benchmark_results('nxd_summary.csv')

### Тензоры размеров (64, ..., 64, n)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# device = 'cpu'
epsilon = 1e-7  # Уровень шума 
n_samples = 1000  # Сэмплы для randomised_parafac

num_repeats_one_tensor = 3  # Повторов для одного и того же тензора (усреднение шума измерений)
num_tensors = 5            # Количество разных тензоров одного размера (усреднение по данным)
max_log = 4                # Максимальная степень двойки
dims = [2, 3, 4]              # Размерности (порядок тензора)

results_summary = [] # Для усредненных данных
results_raw = []     # Для сырых данных всех запусков

for d in dims:
    for n in [2**i for i in range(1, max_log + 1)]: 
        
        # Накопители для усреднения по категории (размер n, порядок d)
        agg_time_cp = []
        agg_time_svd = []
        agg_rank_err_cp = []
        agg_rank_err_svd = []
        
        for t_idx in range(num_tensors):
            
            tensor_shape = tuple([n for _ in range(d)])
            tucker_rank_true = tuple([random.randint(1, tensor_shape[i]) for i in range(d)])
            
            try:
                B_cpu, _, _ = generate_low_tucker_rank_tensor(tensor_shape, tucker_rank_true)
                # noise = torch.randn_like(B_cpu) * epsilon * tl.norm(B_cpu)
                # B_cpu = B_cpu + noise
                
                B = tl.tensor(B_cpu, device=device)
        
                cp_rank = max(tensor_shape)
                
            except Exception as e:
                print(f"Error generating {tensor_shape}: {e}")
                continue

            # Вектор истинного ранга для расчета ошибки
            vec_true = torch.tensor(tucker_rank_true, dtype=torch.float32, device='cpu')

            # Повторные запуски алгоритмов на одном тензоре
            for r_idx in range(num_repeats_one_tensor):
                run_stat = {
                    "Order": d,
                    "Size": n,
                    "Tensor_ID": t_idx,
                    "True_Rank": str(tucker_rank_true)
                }
                
                # CP
                start_event = torch.cuda.Event(enable_timing=True)
                end_event = torch.cuda.Event(enable_timing=True)
                
                time_cp_total = np.nan
                rank_err_cp = np.nan
                calc_rank_cp = None
                
                try:
                    start_event.record()
                    
                    # Разложение
                    factors_obj = randomised_parafac(
                        B, 
                        rank=max(tensor_shape), 
                        n_samples=n_samples, 
                        tol=1e-3, 
                        verbose=0
                    )
                    
                    if hasattr(factors_obj, 'factors'):
                        factors = factors_obj.factors
                    else:
                        # (weights, factors)
                        factors = factors_obj[1]

                    current_calc_rank = []
                    for factor_matrix in factors:
                        # factor_matrix shape: (Size, CP_Rank)
                        # Ранг матрицы <= min(Size, CP_Rank)
                        r_approx = torch.linalg.matrix_rank(factor_matrix).item()
                        current_calc_rank.append(r_approx)
                    
                    calc_rank_cp = tuple(current_calc_rank)
                    
                    end_event.record()
                    torch.cuda.synchronize()
                    time_cp_total = start_event.elapsed_time(end_event) / 1000.0
                    
                    # Ошибка ранга CP
                    vec_calc_cp = torch.tensor(calc_rank_cp, dtype=torch.float32, device='cpu')
                    rank_err_cp = (torch.norm(vec_true - vec_calc_cp) / torch.norm(vec_true)).item()
                    
                except RuntimeError as e:
                    if "out of memory" in str(e):
                        torch.cuda.empty_cache()
                    else:
                        print(f"CP Error: {e}")

                # SVD
                start_event_svd = torch.cuda.Event(enable_timing=True)
                end_event_svd = torch.cuda.Event(enable_timing=True)
                
                time_svd_total = np.nan
                rank_err_svd = np.nan
                calc_rank_svd = None

                try:
                    start_event_svd.record()
                    
                    current_svd_rank = []
                    for mode in range(d):
                        unfolded = tl.unfold(B, mode)
                        r_mode = torch.linalg.matrix_rank(unfolded).item()
                        current_svd_rank.append(r_mode)
                    
                    calc_rank_svd = tuple(current_svd_rank)
                    
                    end_event_svd.record()
                    torch.cuda.synchronize()
                    time_svd_total = start_event_svd.elapsed_time(end_event_svd) / 1000.0
                    
                    # Ошибка ранга SVD
                    vec_calc_svd = torch.tensor(calc_rank_svd, dtype=torch.float32, device='cpu')
                    rank_err_svd = (torch.norm(vec_true - vec_calc_svd) / torch.norm(vec_true)).item()
                    
                except RuntimeError as e:
                    if "out of memory" in str(e):
                        torch.cuda.empty_cache()
                    else: 
                        print(f"SVD Error: {e}")

                # Запись сырых данных
                run_stat.update({
                    "CP_Time": time_cp_total,
                    "CP_Rank_Est": str(calc_rank_cp),
                    "CP_Rank_Err": rank_err_cp,
                    "SVD_Time": time_svd_total,
                    "SVD_Rank_Est": str(calc_rank_svd),
                    "SVD_Rank_Err": rank_err_svd
                })
                results_raw.append(run_stat)
                
                # Добавляем в списки для усреднения (игнорируя NaN)
                if not np.isnan(time_cp_total): agg_time_cp.append(time_cp_total)
                if not np.isnan(rank_err_cp): agg_rank_err_cp.append(rank_err_cp)
                
                if not np.isnan(time_svd_total): agg_time_svd.append(time_svd_total)
                if not np.isnan(rank_err_svd): agg_rank_err_svd.append(rank_err_svd)

        # Агрегация по размеру N (после прохода всех тензоров и повторов)
        summary_stat = {
            "Dimension": d,
            "Size": n,
            "Avg_Time_CP": np.mean(agg_time_cp) if agg_time_cp else np.nan,
            "Std_Time_CP": np.std(agg_time_cp) if agg_time_cp else np.nan,
            "Avg_Rank_Err_CP": np.mean(agg_rank_err_cp) if agg_rank_err_cp else np.nan,
            
            "Avg_Time_SVD": np.mean(agg_time_svd) if agg_time_svd else np.nan,
            "Std_Time_SVD": np.std(agg_time_svd) if agg_time_svd else np.nan,
            "Avg_Rank_Err_SVD": np.mean(agg_rank_err_svd) if agg_rank_err_svd else np.nan,
            
            "Successful_Runs_CP": len(agg_time_cp),
            "Successful_Runs_SVD": len(agg_time_svd)
        }
        results_summary.append(summary_stat)
        
        print(f"Done {n}^({d}): "
              f"CP Time={summary_stat['Avg_Time_CP']:.4f}s, "
              f"SVD Time={summary_stat['Avg_Time_SVD']:.4f}s")
        
        # Очистка памяти перед следующим увеличением размера
        torch.cuda.empty_cache()

# Сохранение результатов
df_raw = pd.DataFrame(results_raw)
df_summary = pd.DataFrame(results_summary)

pd.options.display.float_format = '{:,.4f}'.format

df_summary.to_csv('64xn_summary.csv', sep=';', index=False)
df_raw.to_csv('64xn_all.csv', sep=';', index=False)

print("\n--- Summary Results ---")
print(df_summary[['Dimension', 'Size', 'Avg_Time_CP', 'Avg_Time_SVD', 'Avg_Rank_Err_CP', 'Avg_Rank_Err_SVD']].to_string())
print("\nFiles saved: 'tensor_rank_summary.csv', 'tensor_rank_raw_runs.csv'")


In [ ]:
plot_benchmark_results('64xn_summary.csv')

#### Добавим шум


Воспользуемся VBMF для поиска ранга зашумленных матриц. В svd нужно установить full_matrices = false!!!

In [14]:
import numpy as np
from scipy.sparse.linalg import svds
from scipy.optimize import minimize_scalar

def VBMF(Y, cacb, sigma2=None, H=None):
    """Implementation of the analytical solution to Variational Bayes Matrix Factorization.

    This function can be used to calculate the analytical solution to VBMF. 
    This is based on the paper and MatLab code by Nakajima et al.:
    "Global analytic solution of fully-observed variational Bayesian matrix factorization."

    Notes
    -----
        If sigma2 is unspecified, it is estimated by minimizing the free energy.
        If H is unspecified, it is set to the smallest of the sides of the input Y.
        To estimate cacb, use the function EVBMF().

    Attributes
    ----------
    Y : numpy-array
        Input matrix that is to be factorized. Y has shape (L,M), where L<=M.
        
    cacb : int
        Product of the prior variances of the matrices that factorize the input.
    
    sigma2 : int or None (default=None)
        Variance of the noise on Y.
        
    H : int or None (default = None)
        Maximum rank of the factorized matrices.
        
    Returns
    -------
    U : numpy-array
        Left-singular vectors. 
        
    S : numpy-array
        Diagonal matrix of singular values.
        
    V : numpy-array
        Right-singular vectors.
        
    post : dictionary
        Dictionary containing the computed posterior values.
        
        
    References
    ----------
    .. [1] Nakajima, Shinichi, et al. "Global analytic solution of fully-observed variational Bayesian matrix factorization." Journal of Machine Learning Research 14.Jan (2013): 1-37.
    
    .. [2] Nakajima, Shinichi, et al. "Perfect dimensionality recovery by variational Bayesian PCA." Advances in Neural Information Processing Systems. 2012.
    """    
    
    L,M = Y.shape #has to be L<=M

    if H is None:
        H = L
    
    #SVD of the input matrix, max rank of H
    U,s,V = np.linalg.svd(Y)
    U = U[:,:H]
    s = s[:H]
    V = V[:H].T 

    #Calculate residual
    residual = 0.
    if H<L:
        residual = np.sum(np.sum(Y**2)-np.sum(s**2))

    #Estimation of the variance when sigma2 is unspecified
    if sigma2 is None: 
        upper_bound = (np.sum(s**2)+ residual)/(L+M)

        if L==H: 
            lower_bound = s[-1]**2/M
        else:
            lower_bound = residual/((L-H)*M)

        sigma2_opt = minimize_scalar(VBsigma2, args=(L,M,cacb,s,residual), bounds=[lower_bound, upper_bound], method='Bounded')
        sigma2 = sigma2_opt.x
        print("Estimated sigma2: ", sigma2)

    #Threshold gamma term
    #Formula above (21) from [1]
    thresh_term = (L+M + sigma2/cacb**2)/2 
    threshold = np.sqrt( sigma2 * (thresh_term + np.sqrt(thresh_term**2 - L*M) ))
              
    #Number of singular values where gamma>threshold
    pos = np.sum(s>threshold)

    #Formula (10) from [2]
    d = np.multiply(s[:pos], 
                    1 - np.multiply(sigma2/(2*s[:pos]**2),
                                    L+M+np.sqrt( (M-L)**2 + 4*s[:pos]**2/cacb**2 )))

    #Computation of the posterior
    post = {}
    zeta = sigma2/(2*L*M) * (L+M+sigma2/cacb**2 - np.sqrt((L+M+sigma2/cacb**2)**2 - 4*L*M))
    post['ma'] = np.zeros(H) 
    post['mb'] = np.zeros(H)
    post['sa2'] = cacb * (1-L*zeta/sigma2) * np.ones(H)
    post['sb2'] = cacb * (1-M*zeta/sigma2) * np.ones(H)  

    delta = cacb/sigma2 * (s[:pos]-d- L*sigma2/s[:pos])
    post['ma'][:pos] = np.sqrt(np.multiply(d, delta))
    post['mb'][:pos] = np.sqrt(np.divide(d, delta))
    post['sa2'][:pos] = np.divide(sigma2*delta, s[:pos])
    post['sb2'][:pos] = np.divide(sigma2, np.multiply(delta, s[:pos]))
    post['sigma2'] = sigma2
    post['F'] = 0.5*(L*M*np.log(2*np.pi*sigma2) + (residual+np.sum(s**2))/sigma2 - (L+M)*H
               + np.sum(M*np.log(cacb/post['sa2']) + L*np.log(cacb/post['sb2'])
                        + (post['ma']**2 + M*post['sa2'])/cacb + (post['mb']**2 + L*post['sb2'])/cacb
                        + (-2 * np.multiply(np.multiply(post['ma'], post['mb']), s)
                           + np.multiply(post['ma']**2 + M*post['sa2'],post['mb']**2 + L*post['sb2']))/sigma2))

    return U[:,:pos], np.diag(d), V[:,:pos], post


def VBsigma2(sigma2,L,M,cacb,s,residual):
    H = len(s)

    thresh_term = (L+M + sigma2/cacb**2)/2 
    threshold = np.sqrt( sigma2 * (thresh_term + np.sqrt(thresh_term**2 - L*M) ))
    pos = np.sum(s>threshold)
    
    d = np.multiply(s[:pos], 
                    1 - np.multiply(sigma2/(2*s[:pos]**2),
                                    L+M+np.sqrt( (M-L)**2 + 4*s[:pos]**2/cacb**2 )))

    zeta = sigma2/(2*L*M) * (L+M+sigma2/cacb**2 - np.sqrt((L+M+sigma2/cacb**2)**2 - 4*L*M))
    post_ma = np.zeros(H) 
    post_mb = np.zeros(H)
    post_sa2 = cacb * (1-L*zeta/sigma2) * np.ones(H)
    post_sb2 = cacb * (1-M*zeta/sigma2) * np.ones(H)  

    delta = cacb/sigma2 * (s[:pos]-d- L*sigma2/s[:pos])
    post_ma[:pos] = np.sqrt(np.multiply(d, delta))
    post_mb[:pos] = np.sqrt(np.divide(d, delta))
    post_sa2[:pos] = np.divide(sigma2*delta, s[:pos])
    post_sb2[:pos] = np.divide(sigma2, np.multiply(delta, s[:pos]))

    F = 0.5*(L*M*np.log(2*np.pi*sigma2) + (residual+np.sum(s**2))/sigma2 - (L+M)*H
               + np.sum(M*np.log(cacb/post_sa2) + L*np.log(cacb/post_sb2)
                        + (post_ma**2 + M*post_sa2)/cacb + (post_mb**2 + L*post_sb2)/cacb
                        + (-2 * np.multiply(np.multiply(post_ma, post_mb), s)
                           + np.multiply(post_ma**2 + M*post_sa2,post_mb**2 + L*post_sb2))/sigma2))
    return F


def EVBMF(Y, sigma2=None, H=None):
    """Implementation of the analytical solution to Empirical Variational Bayes Matrix Factorization.

    This function can be used to calculate the analytical solution to empirical VBMF. 
    This is based on the paper and MatLab code by Nakajima et al.:
    "Global analytic solution of fully-observed variational Bayesian matrix factorization."

    Notes
    -----
        If sigma2 is unspecified, it is estimated by minimizing the free energy.
        If H is unspecified, it is set to the smallest of the sides of the input Y.

    Attributes
    ----------
    Y : numpy-array
        Input matrix that is to be factorized. Y has shape (L,M), where L<=M.
    
    sigma2 : int or None (default=None)
        Variance of the noise on Y.
        
    H : int or None (default = None)
        Maximum rank of the factorized matrices.
        
    Returns
    -------
    U : numpy-array
        Left-singular vectors. 
        
    S : numpy-array
        Diagonal matrix of singular values.
        
    V : numpy-array
        Right-singular vectors.
        
    post : dictionary
        Dictionary containing the computed posterior values.
        
        
    References
    ----------
    .. [1] Nakajima, Shinichi, et al. "Global analytic solution of fully-observed variational Bayesian matrix factorization." Journal of Machine Learning Research 14.Jan (2013): 1-37.
    
    .. [2] Nakajima, Shinichi, et al. "Perfect dimensionality recovery by variational Bayesian PCA." Advances in Neural Information Processing Systems. 2012.     
    """   
    L,M = Y.shape #has to be L<=M

    if H is None:
        H = L

    alpha = L/M
    tauubar = 2.5129*np.sqrt(alpha)
    
    #SVD of the input matrix, max rank of H
    U,s,V = np.linalg.svd(Y)
    U = U[:,:H]
    s = s[:H]
    V = V[:H].T 

    #Calculate residual
    residual = 0.
    if H<L:
        residual = np.sum(np.sum(Y**2)-np.sum(s**2))

    #Estimation of the variance when sigma2 is unspecified
    if sigma2 is None: 
        xubar = (1+tauubar)*(1+alpha/tauubar)
        eH_ub = int(np.min([np.ceil(L/(1+alpha))-1, H]))-1
        upper_bound = (np.sum(s**2)+residual)/(L*M)
        lower_bound = np.max([s[eH_ub+1]**2/(M*xubar), np.mean(s[eH_ub+1:]**2)/M])

        scale = 1.#/lower_bound
        s = s*np.sqrt(scale)
        residual = residual*scale
        lower_bound = lower_bound*scale
        upper_bound = upper_bound*scale

        sigma2_opt = minimize_scalar(EVBsigma2, args=(L,M,s,residual,xubar), bounds=[lower_bound, upper_bound], method='Bounded')
        sigma2 = sigma2_opt.x

        print(sigma2)

    #Threshold gamma term
    threshold = np.sqrt(M*sigma2*(1+tauubar)*(1+alpha/tauubar))
    pos = np.sum(s>threshold)

    #Formula (15) from [2]
    d = np.multiply(s[:pos]/2, 1-np.divide((L+M)*sigma2, s[:pos]**2) + np.sqrt((1-np.divide((L+M)*sigma2, s[:pos]**2))**2 -4*L*M*sigma2**2/s[:pos]**4) )

    #Computation of the posterior
    post = {}
    post['ma'] = np.zeros(H) 
    post['mb'] = np.zeros(H)
    post['sa2'] = np.zeros(H) 
    post['sb2'] = np.zeros(H) 
    post['cacb'] = np.zeros(H)  

    tau = np.multiply(d, s[:pos])/(M*sigma2)
    delta = np.multiply(np.sqrt(np.divide(M*d, L*s[:pos])), 1+alpha/tau)

    post['ma'][:pos] = np.sqrt(np.multiply(d, delta))
    post['mb'][:pos] = np.sqrt(np.divide(d, delta))
    post['sa2'][:pos] = np.divide(sigma2*delta, s[:pos])
    post['sb2'][:pos] = np.divide(sigma2, np.multiply(delta, s[:pos]))
    post['cacb'][:pos] = np.sqrt(np.multiply(d, s[:pos])/(L*M))
    post['sigma2'] = sigma2
    post['F'] = 0.5*(L*M*np.log(2*np.pi*sigma2) + (residual+np.sum(s**2))/sigma2 
                     + np.sum(M*np.log(tau+1) + L*np.log(tau/alpha +1) - M*tau))

    return U[:,:pos], np.diag(d), V[:,:pos], post

def EVBsigma2(sigma2,L,M,s,residual,xubar):
    H = len(s)

    alpha = L/M
    x = s**2/(M*sigma2) 

    z1 = x[x>xubar]
    z2 = x[x<=xubar]
    tau_z1 = tau(z1, alpha)

    term1 = np.sum(z2 - np.log(z2))
    term2 = np.sum(z1 - tau_z1)
    term3 = np.sum( np.log( np.divide(tau_z1+1, z1)))
    term4 = alpha*np.sum(np.log(tau_z1/alpha+1))
    
    obj = term1+term2+term3+term4+ residual/(M*sigma2) + (L-H)*np.log(sigma2)

    return obj

def phi0(x):
    return x-np.log(x)

def phi1(x, alpha):
    return np.log(tau(x,alpha)+1) + alpha*np.log(tau(x,alpha)/alpha + 1) - tau(x,alpha)

def tau(x, alpha):
    return 0.5 * (x-(1+alpha) + np.sqrt((x-(1+alpha))**2 - 4*alpha))

Тест VBMF

In [15]:
np.random.seed(99)
true_rank = 55
L, M = 120, 180
noise_level = 0.05

A_true = np.random.randn(L, true_rank)
B_true = np.random.randn(M, true_rank)
Y_clean = A_true @ B_true.T

noise = noise_level * np.linalg.norm(Y_clean) / np.sqrt(L * M) * np.random.randn(L, M)
Y = Y_clean + noise

print(f"Форма матрицы: {Y.shape}")
print(f"Истинный ранг: {true_rank}")
print(f"Уровень шума: {noise_level}")

print("\n--- Запуск EVBMF ---")
U_e, S_e, V_e, post_e = EVBMF(Y)
estimated_rank = U_e.shape[1]

print(f"Найденный ранг: {estimated_rank}")
# print(f"Сингулярные значения (на 2 значения больше,  чем  ранг): {np.diag(S_e)[:estimated_rank + 2].round(4)}")

Y_reconstructed = U_e @ S_e @ V_e.T
recon_error = np.linalg.norm(Y - Y_reconstructed) / np.linalg.norm(Y)
print(f"Относительная ошибка восстановления: {recon_error:.6f}")

Форма матрицы: (120, 180)
Истинный ранг: 55
Уровень шума: 0.05

--- Запуск EVBMF ---
0.23122702866012207
Найденный ранг: 55
Относительная ошибка восстановления: 0.031280


In [18]:
"""Test 4: Сравнение EVBMF и VBMF"""
print("\n" + "="*80)
print("TEST 4: Сравнение EVBMF и VBMF")
print("="*80)

np.random.seed(777)
true_rank = 81
L, M = 500, 500
noise_level = 0

A_true = np.random.randn(L, true_rank)
B_true = np.random.randn(M, true_rank)
Y_clean = A_true @ B_true.T
# print(Y_clean)
noise = noise_level * np.linalg.norm(Y_clean) / np.sqrt(L * M) * np.random.randn(L, M)
Y = Y_clean + noise

print(f"Данные: L={L}, M={M}, ранг={true_rank}, шум={noise_level}\n")

# EVBMF
print("EVBMF (автоматическая оценка cacb)")
U_e, S_e, V_e, post_e = EVBMF(Y)
cacb_evbmf = np.mean(post_e['cacb'][:U_e.shape[1]])
print(f"Найденный ранг: {U_e.shape[1]}")
print(f"Оцененный cacb: {cacb_evbmf:.6f}")
Y_rec_e = U_e @ S_e @ V_e.T
error_e = np.linalg.norm(Y - Y_rec_e) / np.linalg.norm(Y)
print(f"Ошибка восстановления: {error_e:.6f}\n")

# VBMF с найденным cacb
print("VBMF (с cacb из EVBMF)")
U_v, S_v, V_v, post_v = VBMF(Y, cacb=cacb_evbmf)
print(f"Найденный ранг: {U_v.shape[1]}")
Y_rec_v = U_v @ S_v @ V_v.T
error_v = np.linalg.norm(Y - Y_rec_v) / np.linalg.norm(Y)
print(f"Ошибка восстановления: {error_v:.6f}\n")



TEST 4: Сравнение EVBMF и VBMF
Данные: L=500, M=500, ранг=81, шум=0

EVBMF (автоматическая оценка cacb)
6.413009954398928e-06
Найденный ранг: 81
Оцененный cacb: 0.962754
Ошибка восстановления: 0.000000

VBMF (с cacb из EVBMF)
Estimated sigma2:  24.056464132311362
Найденный ранг: 81
Ошибка восстановления: 0.105653



In [23]:
tensorly.set_backend('pytorch')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
# device = 'cpu'

tensor_shape = (500, 500, 500)
tucker_rank = (156, 30, 421)

B, _, __ = generate_low_tucker_rank_tensor(tensor_shape, tucker_rank)
# B = B + 0.001 * torch.randn_like(B)

B = tensorly.tensor(B, device=device)   


factors, errors = randomised_parafac(
    B,
    rank=500,
    tol=1e-3,
    n_samples=1000,
    return_errors= True,
    verbose=0,
)
rank = []
for factor in factors[1]:
    # print(factor.detach().cpu().numpy())

    x = factor.detach().cpu().numpy()
    U_e, S_e, V_e, post_e = EVBMF(x)

    estimated_rank = U_e.shape[1]
    rank.append(estimated_rank)


print(f"Ранг от VBMF: {tuple(rank)}")
# print(f"Ранг от linalg.matrx_rank: {[int(torch.linalg.matrix_rank(factor)) for factor in factors[1]]}")
print(f"Настоящий ранг: {tucker_rank}")


5.732489077405073e-06
3.334172751117026e-06
0.37984453039338717
Ранг от VBMF: (156, 30, 65)
Настоящий ранг: (156, 30, 421)
